# APM_50441_EP - Machine Learning 2 - Lab 4: Unsupervised learning, dimension reduction, clustering and generative models

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/APM-50441-EP/APM_50441_EP-2026-2027/blob/main/PC4/APM_50441_EP_Lab_4_simplified_with_solution.ipynb) or run it on your own computer (see *Working environment* below).

In [ ]:
# On Colab (R runtime) only: installs the packages of this lab (once per session, a few minutes).
# On your own computer, use PC/env/install_packages.R instead (see Working environment).
if (nzchar(Sys.getenv("COLAB_RELEASE_TAG"))) {
  ubuntu <- sub("^VERSION_CODENAME=", "", grep("^VERSION_CODENAME=", readLines("/etc/os-release"), value = TRUE))
  options(repos = c(CRAN = paste0("https://packagemanager.posit.co/cran/__linux__/", ubuntu, "/latest")))  # binaries for this Ubuntu
  source("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/env/install_packages.R")
  install_lab("lab4")
}

This lab illustrates the fourth lecture (*Unsupervised Learning, Generative Learning and More*). Its goals are:

- to see the *curse of dimensionality* numerically (PC4),
- to compute a principal component analysis and to understand it as a reconstruction method,
- to compare nonlinear dimension reduction methods (MDS, Isomap, t-SNE, UMAP) on the digits and on a Swiss roll,
- to compare clustering methods (k-means, Gaussian mixtures, DBSCAN, agglomerative clustering) and to choose the number of groups,
- to generate new digits with a Gaussian mixture, and to train an autoencoder with `{torch}`.

The optional *To go further* part, after the core, can be skipped at first: it covers the rest of the lecture, in particular the generative models (VAE, flows, diffusion and flow matching, GAN).

We use `R` with `tidyverse`.

**Working environment.** Colab is the simplest option: open the notebook version of this lab (`APM_50441_EP_Lab_4.ipynb`, *Open in Colab* link) with the R runtime, and run its first cell, which installs the packages. To work on your own computer, download the lab folder (the `.qmd` file and its files) or clone the course repository, open the `.qmd` file in RStudio or Positron (or VS Code with the R extension), and run it chunk by chunk or render it with Quarto. The packages are installed once with the script `PC/env/install_packages.R` (`source("install_packages.R")`, then `install_lab("lab4")`); `{torch}` downloads its own library the first time (`torch::install_torch()`, done by `install_lab`). The data are read from the local files when they are present (otherwise they are downloaded).

**Use of AI.** AI assistants are allowed, but you must be able to explain every line you use, and you should answer the *predict before you run* questions before running the code: the suggested code may not be correct!

In [ ]:
library(cluster)
library(igraph)
library(dbscan)
library(mclust)
library(Rtsne)
library(uwot)
library(torch)
library(patchwork)
library(tidyverse)   # loaded last, so that purrr::map is not masked by mclust::map
theme_set(theme_light())
if (!file.exists("data/digits.csv")) {
  dir.create("data", showWarnings = FALSE)
  download.file("https://raw.githubusercontent.com/APM-50441-EP/APM_50441_EP-2026-2027/main/PC4/data/digits.csv", "data/digits.csv")
}
digits <- read_csv("data/digits.csv", show_col_types = FALSE)
pixels <- as.matrix(select(digits, -label))
labels <- factor(digits$label)
dim(pixels)

## 1. The curse of dimensionality

PC4 shows that, for $n$ points uniform in $[0,1]^d$, the norms and the pairwise distances concentrate: $\|X\|/\sqrt{d} \to \sqrt{1/3}$ and $\|X - X'\|/\sqrt{d} \to \sqrt{1/6}$, and the ratio between the largest and the smallest distances tends to $1$ as long as $\log(n)/d \to 0$.

**Question** *Predict before you run:* for $250$ points uniform in $[0,1]^d$, what happens to the ratio between the largest and the smallest pairwise distances when $d$ grows? Compute, for $d \in \{2, 10, 100, 500, 1000\}$, the minimum, the maximum and the ratio of the distances to the origin and of the pairwise distances (with `dist()`), divided by $\sqrt{d}$.

**Answer**

In [ ]:
set.seed(42)
curse <- map_dfr(c(2, 10, 100, 500, 1000), function(d) {
  X <- matrix(runif(250 * d), 250, d)
  norms <- sqrt(rowSums(X^2)) / sqrt(d)
  pairwise <- as.vector(dist(X)) / sqrt(d)
  tibble(d = d, `min norm` = min(norms), `max norm` = max(norms), `ratio norms` = max(norms) / min(norms),
         `min distance` = min(pairwise), `max distance` = max(pairwise), `ratio distances` = max(pairwise) / min(pairwise))
})
curse
c(`sqrt(1/3)` = sqrt(1 / 3), `sqrt(1/6)` = sqrt(1 / 6))

**Conclusion** In dimension $2$, the distances between the points vary a lot: the largest pairwise distance is $300$ times the smallest one. When $d$ grows, the norms divided by $\sqrt d$ concentrate around $\sqrt{1/3} \approx 0.577$ (between $0.554$ and $0.595$ for $d = 1000$) and the pairwise distances around $\sqrt{1/6} \approx 0.408$ (between $0.373$ and $0.439$): the ratio between the largest and the smallest distances tends to $1$ ($1.18$ for $d = 1000$). In high dimension, all the points are almost equally far from each other: the "nearest" neighbor is hardly nearer than the others (PC4).

---

**Question** In $[0,1]^d$, a cube that contains a fraction $r$ of the data (uniform) has a side $r^{1/d}$. Plot this side for $r = 1\%$ and $r = 10\%$ as a function of $d \in [1, 100]$. What does it mean for a "local" method such as the $k$-nearest neighbors?

**Answer**

In [ ]:
expand_grid(d = 1:100, r = c(0.01, 0.1)) |>
  mutate(side = r^(1 / d), r = scales::percent(r)) |>
  ggplot(aes(d, side, color = r)) + geom_line(linewidth = 1) + labs(y = "side of the cube", color = "fraction of the data")

**Conclusion** In dimension $1$, $1\%$ of the data is in an interval of length $0.01$; in dimension $10$, a cube containing $1\%$ of the data has a side of $0.63$, and in dimension $100$ of $0.955$: it spans almost the whole range of each feature. A "local" method such as the $k$-NN is then not local any more: either its neighborhoods are large (bias) or they contain very few points (variance). Real data, such as the digits, escape this curse because they lie near a structure of much lower dimension than their ambient space.

---

**Question** Random projections do not suffer from the dimension: by the Johnson–Lindenstrauss lemma, projecting $n$ points on $k = O(\log(n)/\varepsilon^2)$ random directions preserves all their pairwise distances up to a factor $1 \pm \varepsilon$. Project $250$ Gaussian points of dimension $1000$ on $k \in \{10, 50, 200\}$ random Gaussian directions (scaled by $1/\sqrt{k}$), and plot the distribution of the ratios between the projected and the original pairwise distances.

**Answer**

In [ ]:
set.seed(1)
X <- matrix(rnorm(250 * 1000), 250, 1000)
original <- as.vector(dist(X))
projections <- map_dfr(c(10, 50, 200), function(k) {
  W <- matrix(rnorm(1000 * k), 1000, k) / sqrt(k)
  tibble(k = factor(k), ratio = as.vector(dist(X %*% W)) / original)
})
projections |> group_by(k) |> summarise(`1% quantile` = quantile(ratio, 0.01), median = median(ratio), `99% quantile` = quantile(ratio, 0.99))
ggplot(projections, aes(ratio, fill = k)) + geom_density(alpha = 0.4)

**Conclusion** The ratios between the projected and the original distances concentrate around $1$ as $k$ grows: $98\%$ of them are between $0.50$ and $1.49$ for $k = 10$, between $0.77$ and $1.24$ for $k = 50$ and between $0.88$ and $1.12$ for $k = 200$. The width decreases like $1/\sqrt k$ and does not depend on the original dimension ($1000$): this is the Johnson–Lindenstrauss lemma, $k = O(\log(n)/\varepsilon^2)$ random directions preserve all the pairwise distances of $n$ points within a factor $1 \pm \varepsilon$. Random projections are a cheap first reduction, useful before a $k$-NN search for instance.

---

## 2. Principal component analysis of the digits

The digits are $8 \times 8$ grayscale images (values from $0$ to $16$): $1797$ observations in dimension $64$. A PCA looks for the linear subspace of dimension $q$ that keeps the largest inertia, or equivalently that reconstructs the data with the smallest error (PC4); it is computed by a singular value decomposition of the centered data matrix.

**Question** Compute the PCA of the digits with `prcomp` (without scaling), and check that its scores are the left singular vectors of the centered matrix multiplied by the singular values (`svd`), and that classical multidimensional scaling (`cmdscale`) of the Euclidean distances gives the same coordinates (up to signs). Plot the proportion of explained variance and the first two scores colored by the digit.

**Answer**

In [ ]:
pca <- prcomp(pixels)
centered <- scale(pixels, scale = FALSE)
s <- svd(centered)
mds <- cmdscale(dist(pixels), k = 2)
c(`max |prcomp - svd|` = max(abs(abs(pca$x[, 1:5]) - abs(s$u[, 1:5] %*% diag(s$d[1:5])))),
  `max |prcomp - cmdscale|` = max(abs(abs(pca$x[, 1:2]) - abs(mds))))
explained <- tibble(component = 1:64, proportion = pca$sdev^2 / sum(pca$sdev^2), cumulative = cumsum(proportion))
explained |> filter(component %in% c(2, 5, 10, 20, 30))
(ggplot(explained, aes(component, cumulative)) + geom_line() + geom_point(size = 0.8) + labs(y = "cumulative explained variance")) +
  (ggplot(tibble(PC1 = pca$x[, 1], PC2 = pca$x[, 2], digit = labels), aes(PC1, PC2, color = digit)) + geom_point(size = 0.6))

**Conclusion** The scores of `prcomp` are those of the SVD of the centered matrix, and classical MDS on the Euclidean distances gives the same coordinates, up to signs (differences of order $10^{-12}$): PCA, the SVD and classical MDS are the same method. The first two components explain only $28.5\%$ of the variance, $10$ components $74\%$ and $30$ components $96\%$. On the first plane, some digits form separate groups ($0$, $4$, $6$) while others overlap: a linear projection on $2$ dimensions keeps too little information to separate the $10$ classes.

---

**Question** Plot the mean digit and the first $6$ principal directions as $8\times 8$ images ("eigen-digits"). Then plot the reconstruction of the first $5$ digits with $q \in \{2, 5, 10, 20\}$ components, and the mean reconstruction error as a function of $q$. Relate this error to the explained variance.

**Answer**

In [ ]:
image_tiles <- function(vectors, titles, nrow = 1) {
  map2_dfr(seq_len(nrow(vectors)), titles, \(i, t) tibble(title = t, row = rep(1:8, each = 8), col = rep(1:8, 8), value = vectors[i, ])) |>
    mutate(title = factor(title, levels = unique(titles))) |>
    ggplot(aes(col, -row, fill = value)) + geom_raster() + facet_wrap(~ title, nrow = nrow) +
    scale_fill_gradient2() + coord_equal() + theme_void() + theme(legend.position = "none")
}
tiles <- rbind(pca$center, t(pca$rotation[, 1:6]))
image_tiles(tiles / apply(abs(tiles), 1, max), c("mean", paste0("PC", 1:6)))   # each image on its own scale

In [ ]:
reconstruct <- function(q) sweep(pca$x[, 1:q, drop = FALSE] %*% t(pca$rotation[, 1:q, drop = FALSE]), 2, pca$center, "+")
map(c(2, 5, 10, 20), \(q) image_tiles(reconstruct(q)[1:5, ], paste0("digit ", 1:5, ", q = ", q))) |> wrap_plots(ncol = 1)
errors <- tibble(q = 1:40, `mean squared error` = map_dbl(q, \(q) mean(rowSums((pixels - reconstruct(q))^2))),
                 `sum of the remaining eigenvalues` = map_dbl(q, \(q) sum(pca$sdev[(q + 1):64]^2) * (nrow(pixels) - 1) / nrow(pixels)))
errors |> filter(q %in% c(2, 5, 10, 20))

**Conclusion** The mean digit is a blurred average shape, and the principal directions (each image on its own color scale) are contrasts between regions of the image, with positive and negative pixels: adding or subtracting them moves strokes. With $q = 2$ components, all the reconstructions look alike; with $q = 10$, the digits become recognizable; with $q = 20$, they are close to the original images. The mean squared reconstruction error equals the sum of the eigenvalues of the discarded components ($859$ for $q = 2$, $127$ for $q = 20$): the PCA is the best linear reconstruction of rank $q$ (Eckart–Young, PC4), and the proportion of explained variance is $1$ minus the relative reconstruction error.

---

## 3. Nonlinear dimension reduction

A PCA can only find linear structures. On a *Swiss roll*, a 2D sheet rolled in 3D, the interesting coordinate is the position along the roll, which is not linear. *Isomap* replaces the Euclidean distances by *geodesic* distances (shortest paths in a neighborhood graph) and applies MDS to them; *t-SNE* and *UMAP* preserve the neighborhoods rather than the distances.

In [ ]:
set.seed(3)
t_roll <- 1.5 * pi * (1 + 2 * runif(1500))
swiss <- cbind(t_roll * cos(t_roll), 20 * runif(1500), t_roll * sin(t_roll))

**Question** Implement Isomap: build the graph of the $10$ nearest neighbors (`dbscan::kNN`), compute the shortest path distances (`igraph::distances`) and apply `cmdscale`. Compare the PCA and the Isomap embeddings of the Swiss roll, colored by the position along the roll `t_roll`.

**Answer**

In [ ]:
isomap <- function(X, k = 10, dim = 2) {
  neighbors <- kNN(X, k = k)
  edges <- tibble(from = rep(seq_len(nrow(X)), k), to = as.vector(neighbors$id), weight = as.vector(neighbors$dist))
  graph <- graph_from_data_frame(edges, directed = FALSE, vertices = tibble(id = seq_len(nrow(X))))
  cmdscale(distances(graph), k = dim)
}
swiss_embeddings <- bind_rows(
  tibble(method = "PCA", as_tibble(prcomp(swiss)$x[, 1:2], .name_repair = ~ c("x", "y")), t = t_roll),
  tibble(method = "Isomap", as_tibble(isomap(swiss), .name_repair = ~ c("x", "y")), t = t_roll))
ggplot(swiss_embeddings, aes(x, y, color = t)) + geom_point(size = 0.5) + facet_wrap(~ method, scales = "free") + scale_color_viridis_c()

**Conclusion** The PCA of the Swiss roll only shows the roll seen from the side: the points with very different positions along the roll ($t$) are projected close to each other. Isomap unrolls it: its first coordinate follows $t$ almost monotonically, because the geodesic distances along the neighborhood graph measure the distances *along* the surface, not through the ambient space.

---

To compare embeddings, we use the proportion of the $10$ nearest neighbors of each point (in the original space) that are still among its $10$ nearest neighbors in the embedding:

In [ ]:
knn_preservation <- function(X, embedding, k = 10) {
  original <- kNN(X, k = k)$id
  embedded <- kNN(embedding, k = k)$id
  mean(map_dbl(seq_len(nrow(X)), \(i) length(intersect(original[i, ], embedded[i, ])) / k))
}

**Question** Embed the digits in 2D with PCA, Isomap, t-SNE (`Rtsne`, on the first $50$ principal components, perplexity $30$) and UMAP (`uwot::umap`). Plot the embeddings colored by the digit, and compare their neighbor preservation. Then try the perplexities $5$ and $100$ for t-SNE.

**Answer**

In [ ]:
set.seed(4)
pcs50 <- pca$x[, 1:50]
embeddings <- list(PCA = pca$x[, 1:2], Isomap = isomap(pixels),
                   `t-SNE (perplexity 30)` = Rtsne(pcs50, perplexity = 30, pca = FALSE)$Y,
                   UMAP = umap(pixels, n_neighbors = 15, min_dist = 0.1))
imap_dfr(embeddings, \(e, name) tibble(method = name, x = e[, 1], y = e[, 2], digit = labels)) |>
  mutate(method = factor(method, levels = names(embeddings))) |>
  ggplot(aes(x, y, color = digit)) + geom_point(size = 0.4) + facet_wrap(~ method, scales = "free") +
  guides(color = guide_legend(override.aes = list(size = 3)))

In [ ]:
embeddings[["t-SNE (perplexity 5)"]] <- Rtsne(pcs50, perplexity = 5, pca = FALSE)$Y
embeddings[["t-SNE (perplexity 100)"]] <- Rtsne(pcs50, perplexity = 100, pca = FALSE)$Y
tibble(method = names(embeddings), `10-NN preservation` = map_dbl(embeddings, \(e) knn_preservation(pixels, e)))

**Conclusion** t-SNE and UMAP separate the $10$ digits into well-defined clusters, while the PCA and Isomap show overlapping clouds (Isomap separates some digits, such as $0$, $4$ and $6$). The neighbor preservation measures it: only $12\%$ of the $10$ nearest neighbors are kept by the PCA and $18\%$ by Isomap, against $59\%$ by t-SNE and $50\%$ by UMAP. t-SNE and UMAP are designed to preserve the neighborhoods, at the price of the global distances (the distances between clusters and their sizes in the maps are not meaningful). The perplexity changes the balance: $5$ and $100$ give a slightly lower neighbor preservation ($0.54$ and $0.53$) than $30$.

---

## 4. Clustering

We compare the clustering methods of the lecture on three toy datasets with known groups, and on the digits. The agreement between a clustering and the true groups is measured with the *adjusted Rand index* (ARI): the proportion of pairs of points on which the two partitions agree (both in the same group, or both in different groups), corrected so that a random partition gets $0$ on average and identical partitions get $1$ (`mclust::adjustedRandIndex`).

In [ ]:
set.seed(5)
toy <- list(
  blobs = tibble(x = c(rnorm(200, 0), rnorm(200, 4), rnorm(200, 2)), y = c(rnorm(200, 0), rnorm(200, 0), rnorm(200, 3.5)), group = rep(1:3, each = 200)),
  elongated = tibble(x = c(rnorm(300, 0, 3), rnorm(300, 0, 3)), y = c(rnorm(300, 0, 0.3), rnorm(300, 2, 0.3)), group = rep(1:2, each = 300)),
  rings = tibble(angle = runif(600, 0, 2 * pi), radius = rep(c(1, 3), each = 300) + rnorm(600, 0, 0.15), group = rep(1:2, each = 300)) |>
    transmute(x = radius * cos(angle), y = radius * sin(angle), group))
bind_rows(toy, .id = "dataset") |> ggplot(aes(x, y, color = factor(group))) + geom_point(size = 0.5) +
  facet_wrap(~ dataset, scales = "free") + theme(aspect.ratio = 1) + labs(color = "group")

**Question** On each toy dataset, run $k$-means (`nstart = 20`), a Gaussian mixture (`Mclust`, with the true number of groups), DBSCAN (`dbscan`, $\varepsilon = 0.5$, `minPts = 10`; its noise points form a group of their own) and agglomerative clustering with the single, average and Ward linkages (`hclust` + `cutree`, true number of groups). Report the ARI of each method. Which assumptions explain the results?

**Answer**

In [ ]:
clusterings <- function(data) {
  X <- as.matrix(select(data, x, y)); K <- n_distinct(data$group); D <- dist(X)
  list(`k-means` = kmeans(X, K, nstart = 20)$cluster, `Gaussian mixture` = Mclust(X, G = K, verbose = FALSE)$classification,
       DBSCAN = dbscan(X, eps = 0.5, minPts = 10)$cluster, `single linkage` = cutree(hclust(D, "single"), K),
       `average linkage` = cutree(hclust(D, "average"), K), Ward = cutree(hclust(D, "ward.D2"), K))
}
toy_results <- imap_dfr(toy, \(data, name) imap_dfr(clusterings(data), \(cl, method) tibble(dataset = name, method = method,
                                                                                            ARI = adjustedRandIndex(cl, data$group))))
toy_results |> pivot_wider(names_from = dataset, values_from = ARI)

**Conclusion** Each method succeeds where its assumptions hold. On the blobs (compact and roughly spherical groups), $k$-means and the Gaussian mixture find the groups (ARI $0.90$), Ward and the average linkage almost as well, while the single linkage fails (it chains the touching groups). On the elongated groups, only the Gaussian mixture, which models the covariances, is perfect; DBSCAN finds them ($0.79$, the remaining errors being points labeled as noise). On the rings, only the methods based on connectivity (DBSCAN, single linkage) succeed, as the groups are neither compact nor convex. There is no universal clustering: the notion of group is part of the method.

---

**Question** On the first $10$ principal components of the digits, choose the number of groups $K \in \{2, \dots, 15\}$ with the *elbow* of the within-cluster inertia of $k$-means, the average *silhouette* of $k$-means, and the BIC of Gaussian mixtures with diagonal covariances (`Mclust(..., modelNames = "VVI")`). Compare the chosen numbers with the $10$ digits, and the ARI of the clusterings with the labels.

**Answer**

In [ ]:
pcs10 <- pca$x[, 1:10]
set.seed(6)
D10 <- dist(pcs10)
choice <- map_dfr(2:15, function(K) {
  km <- kmeans(pcs10, K, nstart = 10)
  tibble(K = K, inertia = km$tot.withinss, silhouette = mean(silhouette(km$cluster, D10)[, 3]), ARI = adjustedRandIndex(km$cluster, labels))
})
bic <- Mclust(pcs10, G = 2:15, modelNames = "VVI", verbose = FALSE)
choice <- choice |> mutate(BIC = as.vector(bic$BIC[as.character(2:15), "VVI"]))
choice |> pivot_longer(-K) |> ggplot(aes(K, value)) + geom_line() + geom_point() + facet_wrap(~ name, scales = "free_y")
c(`best silhouette` = choice$K[which.max(choice$silhouette)], `best BIC` = bic$G,
  `ARI of k-means with K = 10` = choice$ARI[choice$K == 10], `ARI of the mixture with K = 10` =
    adjustedRandIndex(Mclust(pcs10, G = 10, modelNames = "VVI", verbose = FALSE)$classification, labels))

**Conclusion** The inertia decreases with $K$ without a clear elbow; the silhouette is maximal for $K = 9$, close to the $10$ digits, and the BIC of the diagonal Gaussian mixtures keeps increasing up to $K = 15$ (the largest value tried): the clusters of digits are not Gaussian, and the mixture uses several components per digit. With $K = 10$, $k$-means agrees reasonably with the labels (ARI $0.65$), better than the mixture ($0.53$). The "right" number of clusters depends on the criterion, and the classes of a supervised problem are only one of the possible structures.

---

## 5. A first generative model: Gaussian mixtures

A density model can be used to *generate* new data: a Gaussian mixture is sampled by drawing a component, then a Gaussian vector. In dimension $64$, the covariances of a mixture are hard to estimate, so we fit the mixture on the first principal components and map the samples back to the pixels.

**Question** Fit a Gaussian mixture with $12$ components and full covariances (`Mclust(..., G = 12, modelNames = "VVV")`) on the first $15$ principal components of the digits, draw $30$ new points with `mclust::sim`, map them back to the pixel space and plot them. Compare with a mixture on $5$ components only.

**Answer**

In [ ]:
set.seed(7)
generate <- function(q, n = 30) {
  fit <- Mclust(pca$x[, 1:q], G = 12, modelNames = "VVV", verbose = FALSE)
  samples <- sim(fit$modelName, fit$parameters, n = n)[, -1]
  pmin(pmax(sweep(samples %*% t(pca$rotation[, 1:q]), 2, pca$center, "+"), 0), 16)
}
(image_tiles(generate(15)[1:12, ], as.character(1:12)) + ggtitle("mixture on 15 principal components")) /
  (image_tiles(generate(5)[1:12, ], as.character(1:12)) + ggtitle("mixture on 5 principal components"))

**Conclusion** The samples of the Gaussian mixture on $15$ principal components look like plausible, slightly blurred digits, of various classes: a simple generative model already captures much of the variability of the digits. With only $5$ principal components, the samples are blurrier and some are ambiguous: the model can only generate what lies in its $5$-dimensional subspace. The quality of a generative model depends both on its density model and on the representation in which it is fitted.

---

## 6. Autoencoders with `{torch}`

An autoencoder is a neural network trained to reconstruct its input through a low-dimensional *code*: an encoder $x \mapsto z$ and a decoder $z \mapsto \hat x$, trained to minimize $\|x - \hat x\|^2$. With linear layers and the squared loss, the best autoencoder spans the same subspace as the PCA; nonlinear layers can find curved structures.

In [ ]:
torch_set_num_threads(2)                            # small networks: a few threads are faster than many
X_scaled <- scale(pixels, scale = FALSE) / 16       # centered, in [-1, 1]
x_tensor <- torch_tensor(X_scaled, dtype = torch_float())
train_autoencoder <- function(encoder, decoder, epochs = 1500, lr = 1e-2, seed = 0) {
  torch_manual_seed(seed)
  parameters <- c(encoder$parameters, decoder$parameters)
  optimizer <- optim_adam(parameters, lr = lr)
  losses <- numeric(epochs)
  for (epoch in seq_len(epochs)) {
    optimizer$zero_grad()
    loss <- nnf_mse_loss(decoder(encoder(x_tensor)), x_tensor)
    loss$backward()
    optimizer$step()
    losses[epoch] <- loss$item()
  }
  losses
}

**Question** Train a *linear* autoencoder $64 \to 2 \to 64$ (`nn_linear` layers, full batch, Adam, $1500$ epochs), and compare its reconstruction error with the PCA with $2$ components. Check that the subspace spanned by the decoder weights is the one of the first two principal directions, by computing the cosines of the *principal angles* between the two subspaces (the singular values of $Q_1^\top Q_2$, where $Q_1$ and $Q_2$ are orthonormal bases).

**Answer**

In [ ]:
torch_manual_seed(0)
encoder <- nn_linear(64, 2)
decoder <- nn_linear(2, 64)
losses <- train_autoencoder(encoder, decoder)
decoder_weights <- as.matrix(decoder$weight)   # 64 x 2
cosines <- svd(t(qr.Q(qr(decoder_weights))) %*% pca$rotation[, 1:2])$d
c(`linear autoencoder MSE` = tail(losses, 1),
  `PCA (q = 2) MSE` = mean((X_scaled - (X_scaled %*% pca$rotation[, 1:2]) %*% t(pca$rotation[, 1:2]))^2),
  `cosine of the principal angles` = cosines)

**Conclusion** The linear autoencoder reaches exactly the reconstruction error of the PCA with $2$ components ($0.0524$), and the cosines of the principal angles between the subspace of its decoder and the first principal plane are $1$: it has found the same subspace (with another basis, since the loss only depends on the subspace). A linear autoencoder trained by gradient descent is a (slow) way to compute a PCA.

---

**Question** Train a deep autoencoder $64 \to 128 \to 32 \to 2 \to 32 \to 128 \to 64$ with `tanh` activations. Compare its reconstruction error with the linear one, and plot the 2D codes of both, colored by the digit.

**Answer**

In [ ]:
torch_manual_seed(0)
deep_encoder <- nn_sequential(nn_linear(64, 128), nn_tanh(), nn_linear(128, 32), nn_tanh(), nn_linear(32, 2))
deep_decoder <- nn_sequential(nn_linear(2, 32), nn_tanh(), nn_linear(32, 128), nn_tanh(), nn_linear(128, 64))
deep_losses <- train_autoencoder(deep_encoder, deep_decoder, epochs = 2000, lr = 5e-3)
c(`deep autoencoder MSE` = tail(deep_losses, 1), `linear autoencoder MSE` = tail(losses, 1))
codes <- bind_rows(tibble(model = "linear", as_tibble(as.matrix(encoder(x_tensor)), .name_repair = ~ c("z1", "z2")), digit = labels),
                   tibble(model = "deep", as_tibble(as.matrix(deep_encoder(x_tensor)), .name_repair = ~ c("z1", "z2")), digit = labels))
ggplot(codes, aes(z1, z2, color = digit)) + geom_point(size = 0.4) + facet_wrap(~ model, scales = "free") +
  guides(color = guide_legend(override.aes = list(size = 3)))

**Conclusion** With nonlinear layers, the reconstruction error with a $2$-dimensional code is three times smaller ($0.018$ against $0.052$): the code can follow a curved structure. The codes of the deep autoencoder separate the digits much better than the linear ones, each digit occupying a branch of a star-shaped map. The autoencoder is a nonlinear dimension reduction, and its decoder a generator of images (but not a probabilistic one: see the VAE).

---

## About this lab

**Use of AI.** Artificial intelligence tools have been used to revise the content of this lab. The main contributor takes full responsibility for this content.

**License.** The texts of this lab are under the [Creative Commons Attribution-ShareAlike 4.0](https://creativecommons.org/licenses/by-sa/4.0/) license (CC BY-SA 4.0), and its code under the [MIT](https://opensource.org/license/mit) license; the datasets keep their own licenses. Main contributor: E. Le Pennec. Contributors: S. Boucheron, A. Dieuleveut, A.K. Fermin, S. Gadat, S. Gaiffas, B. Goujaud, A. Guilloux, Ch. Keribin, Th. Klein, E. Matzner, J. Read, M. Sangnier, E. Scornet.